In [22]:
import numpy as np
import random

n = 14
k = 10

solution_1d = np.array([random.choice([0,1]) for i in range(n) for j in range(k)])
solution_1d

array([0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0,
       0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 1, 1, 0, 0])

In [ ]:
from pymoo import Problem as MyProblem
from enum import Enum
import networkx as nx

ROUTES_KEY = 'routes'
WEIGHT_KEY = 'weight' # or time_min or travel_time

class FitnessType(Enum):
    A = 'Моя функция А которая хочет быть мин'
    B = 'Моя функция B которая тоже хочет быть мин'

class Problem(PymooProblem):

    def __init__(
            self, 
            graph : nx.DiGraph,
            max_routes : int,
            matrix
        ):  
        n = len(graph.nodes)
        k = max_routes
        self.n = n
        self.k = max_routes
        self.graph = self._preprocess_graph(graph)
        self.matrix = matrix
        super().__init__(
            n_var=n*k, # количество переменных в решении
            n_obj=len(list(FitnessType)), # количество оптимизируемых функций
            n_constr=0,
            xl=0,
            xu=1,
            type_var=int # Дискретные переменные (метки)
        )

    def _preprocess_graph(self, graph : nx.Graph):
        graph = graph.copy()
        for u,v,d in graph.edges(data=True):
            d[ROUTES_KEY] = []
        return graph

    def _solution_1d_to_2d(self, solution_1d : np.array) -> np.array:
        solution_2d = np.array([[round(solution_1d[i+j*self.k]) for i in range(self.n)] for j in range(self.k)])
        return solution_2d

    def _evaluate_a(self, graph : nx.Graph) -> float: # минимизируем количество существующих рутов
        unique_routes = set()
        for u,v,d in graph.edges(data=True):
            for route in d[ROUTES_KEY]:
                unique_routes.add(route)
        return len(unique_routes)

    def _evaluate_b(self, graph : nx.Graph) -> float: # здесь особое внимание, тоже минимизируется
        mx = self.matrix
        # в каком-то виде надо оценить граф на предмет того что он как-то адекватно матрицу удовлетворяет
        ...

    def _evaluate_fitness(self, fitness_type : FitnessType, graph : nx.Graph): # плюс минус можешь тут работать точнее начиная отсюда
        if fitness_type == FitnessType.A:
            return self._evaluate_a(graph)
        if fitness_type == FitnessType.B:
            return self._evaluate_b(graph)
        return 0

    def _evaluate_fitnesses(self, graph : nx.Graph):
        return {fitness_type : self._evaluate_fitness(fitness_type, graph) for fitness_type in list(FitnessType)}

    def _solution_to_graph(self, solution_1d : np.array) -> nx.Graph:
        solution_2d = self._solution_1d_to_2d(solution_1d)

        def row_to_nodes(row : np.array):
            nodes_to_visit = [i for i,v in enumerate(row) if v==1]
            return nodes_to_visit

        routes_nodes = [row_to_nodes(row) for row in solution_2d]

        graph = self.graph.copy()

        def nodes_to_route(nodes : list[int]) -> list[int]:
            # тут решается ТСП на этих нодах на твоем графе G*'
            return nx.algorithms.approximation.traveling_salesman_problem(
                graph, 
                weight=WEIGHT_KEY, 
                nodes=nodes
            )
                
        # тут какие мы ноды должны посетить каждым маршрутом из k маршрутов
        routes = [nodes_to_route(route_nodes) for route_nodes in routes_nodes]

        # далее чето делаешь короче с графом чтоб пометить что через него проходит твой маршрут
        for i,route in enumerate(routes):
            for j in range(len(route)-1):
                node_from = route[j]
                node_to = route[j+1]
                graph.edges[node_from, node_to][ROUTES_KEY].append(i)
        return graph

    def _evaluate(self, solutions, out, *args, **kwargs):
        # обязательный метод в пиму, он в него стучит когда проверяет решения и ждет что ты ему отдашь обратно чиселки
        # solutions -- это твое поколение, по сути набор из k особей, каждая из которых имеет n генов
        # каждый ген -- 0 или 1, в зависимости от того едем мы в эту остановку или нет
        # соответственно в графе ноды надо нумеровать с 0

        solutions_fitnesses = {fitness_type : [] for fitness_type in list(FitnessType)} # это тоже не меняй

        for solution in solutions:

            graph = self._solution_to_graph(solution)

            fitnesses = self._evaluate_fitnesses(graph) # и это тоже

            for ft, value in fitnesses.items(): # тоже можешь не трогать
                solutions_fitnesses[ft].append(value) # тоже

        out["F"] = np.column_stack([v for v in solutions_fitnesses.values()]) # это не меняй

IndentationError: expected an indented block after function definition on line 29 (1594472495.py, line 31)

In [ ]:
problem = Problem(graph, k, ndmfndkfnsdnfl)
algorithm = NSGA2(pop_size=10)
res = minimize(problem, algorithm ('n_gen', 100), seed=42, verbose=True)

In [ ]:
res.X # здесь получается массив лучших солюшенов

In [ ]:
res.F # здесь массив фитнессов этих лучших солюшенов